# 01 - Ingestão da Camada Bronze

## Objetivo

Este notebook realiza a ingestão dos arquivos brutos do dataset
Brazilian E-Commerce Public Dataset by Olist para a camada Bronze
do Lakehouse.

Os arquivos originais são mantidos no Volume do Unity Catalog e
persistidos como tabelas Delta na camada Bronze.

Nesta etapa não são aplicadas regras de limpeza ou transformação
de negócio. São adicionados apenas metadados técnicos para garantir
rastreabilidade e auditoria da ingestão

In [0]:
from pyspark.sql.functions import current_timestamp
from pyspark.sql.functions import current_timestamp, col

CATALOG = "datalake_mvp"
SCHEMA_BRONZE = "mvp_bronze"

RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/raw_files"

In [0]:
arquivos = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv"
}

In [0]:
resultados = []

for tabela, arquivo in arquivos.items():

    caminho = f"{RAW_PATH}/{arquivo}"

    print(f"Iniciando ingestão: {arquivo}")

    reader = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("encoding", "UTF-8")
        .option("quote", '"')
        .option("escape", '"')
    )

    # O arquivo de avaliações contém campos textuais
    # que podem possuir quebras de linha.
    if tabela == "order_reviews":
        reader = reader.option("multiLine", True)

    df = reader.csv(caminho)

    df_bronze = (
        df
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_file", col("_metadata.file_path"))
    )

    tabela_destino = f"{CATALOG}.{SCHEMA_BRONZE}.{tabela}"

    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela_destino)
    )

    quantidade = df_bronze.count()

    resultados.append(
        (tabela, arquivo, quantidade)
    )

    print(f"✓ {tabela}: {quantidade:,} registros carregados")

Iniciando ingestão: olist_customers_dataset.csv
✓ customers: 99,441 registros carregados
Iniciando ingestão: olist_geolocation_dataset.csv
✓ geolocation: 1,000,163 registros carregados
Iniciando ingestão: olist_order_items_dataset.csv
✓ order_items: 112,650 registros carregados
Iniciando ingestão: olist_order_payments_dataset.csv
✓ order_payments: 103,886 registros carregados
Iniciando ingestão: olist_order_reviews_dataset.csv
✓ order_reviews: 99,224 registros carregados
Iniciando ingestão: olist_orders_dataset.csv
✓ orders: 99,441 registros carregados
Iniciando ingestão: olist_products_dataset.csv
✓ products: 32,951 registros carregados
Iniciando ingestão: olist_sellers_dataset.csv
✓ sellers: 3,095 registros carregados
Iniciando ingestão: product_category_name_translation.csv
✓ product_category_translation: 71 registros carregados


In [0]:
df_resultados = spark.createDataFrame(
    resultados,
    ["tabela", "arquivo_origem", "quantidade_registros"]
)

display(df_resultados.orderBy("tabela"))

tabela,arquivo_origem,quantidade_registros
customers,olist_customers_dataset.csv,99441
geolocation,olist_geolocation_dataset.csv,1000163
order_items,olist_order_items_dataset.csv,112650
order_payments,olist_order_payments_dataset.csv,103886
order_reviews,olist_order_reviews_dataset.csv,99224
orders,olist_orders_dataset.csv,99441
product_category_translation,product_category_name_translation.csv,71
products,olist_products_dataset.csv,32951
sellers,olist_sellers_dataset.csv,3095


In [0]:
display(
    spark.sql("""
        SHOW TABLES
        IN datalake_mvp.mvp_bronze
    """)
)

database,tableName,isTemporary
mvp_bronze,customers,false
mvp_bronze,geolocation,false
mvp_bronze,order_items,false
mvp_bronze,order_payments,false
mvp_bronze,order_reviews,false
mvp_bronze,orders,false
mvp_bronze,product_category_translation,false
mvp_bronze,products,false
mvp_bronze,sellers,false


## Resultado da Ingestão

A ingestão da camada Bronze foi concluída com sucesso para os nove
arquivos que compõem o dataset da Olist.

Os arquivos originais foram preservados no Volume do Unity Catalog e
persistidos como tabelas Delta no schema `datalake_mvp.mvp_bronze`.

Nesta etapa não foram realizadas transformações relacionadas à qualidade
ou às regras de negócio. Foram adicionados somente metadados técnicos
de ingestão para permitir rastreabilidade da origem dos registros.

A análise e o tratamento dos problemas de qualidade serão realizados
nas etapas subsequentes do pipeline.